# Week 9 guided practice

CML3113. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv and classification_case.csv into Colab with the Files panel before running the cells. The target in this notebook is crusher energy demand in kWh/t.

## Today’s learning

*PowerPoint slide 2*

WHAT WE WILL DO
Turn a number into a yes or no question, teach a model to answer it, and score its answers. Learn the words class, probability, threshold, confusion matrix, precision, recall and F1.

HOW YOU WILL SHOW IT
You can explain why a model that never raises an alert can score 59 percent, and say what a missed alert costs. You can read a confusion matrix and the scores built from it.

## The supplied file

*PowerPoint slide 4*

Extract this week’s student ZIP. Upload classification_case.csv and engineering_data.csv into Colab using the Files panel, or keep them beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.

## Classification turns a number into a yes or no

*PowerPoint slide 5*

Same dots. On the left we predict the number. On the right we predict which side of the limit it falls on.

## Building the alert label

*PowerPoint slide 6*

The first six lines repeat Week 8: read, drop copies, keep valid rows.
>= compares each target with the limit, 13.5 kWh/t. The answer is True or False.
.astype(int) turns True and False into 1 and 0. The training rows are E1 to E4.
sum() adds the ones, so it counts the alerts in the training rows.

The limit is 13.5 kWh/t, a teaching assumption. The counts use the training rows only.

In [ ]:
import pandas as pd
raw = pd.read_csv("engineering_data.csv")
data = raw.drop_duplicates("record_id")
has_target = data["target"].notna()
valid_load = data["load_pct"].between(0, 100)
data = data[has_target & valid_load].copy()
data["alert"] = (data["target"] >= 13.5).astype(int)
groups = ["E1", "E2", "E3", "E4"]
train = data[data["equipment_id"].isin(groups)]
print(len(train), train["alert"].sum())
print(train["alert"].value_counts().to_dict())

**Example output**

```text
58 24
{0: 34, 1: 24}
```

## A model that always says no scores 59 percent

*PowerPoint slide 7*

Most rows are no alert, so a model that never raises an alert still looks accurate.

## Logistic regression turns a line into a probability

*PowerPoint slide 8*

The curve gives a probability between 0 and 1 for every load. Higher load, higher probability of an alert.

## Features on different scales need a common ruler

*PowerPoint slide 9*

scaled value = (value − mean) ÷ standard deviation
Training mean and standard deviation: load_pct 59.9 and 22.7; ambient_c 26.3 and 4.6.
StandardScaler learns these from the training rows only.

## Splitting the rows and choosing features

*PowerPoint slide 10*

train was built on the last slide from groups E1 to E4. E5 validates. E6 stays sealed.
X holds the input columns. y holds the answers.
This time the answer column is alert.

X is a table of features. y is the column of answers.

In [ ]:
valid = data[data["equipment_id"] == "E5"]
features = ["load_pct", "ambient_c"]
X_train, y_train = train[features], train["alert"]
X_valid, y_valid = valid[features], valid["alert"]
print(len(train), len(valid))

**Example output**

```text
58 15
```

## Fitting a classifier and reading its probabilities

*PowerPoint slide 11*

make_pipeline lists the steps in order: fill blanks, scale, classify.
predict_proba returns two columns: the chance of class 0 and of class 1.
[:, 1] keeps the second column, the probability of an alert.
[:5] shows the first five validation rows.

fit learns from the training rows. predict_proba answers for rows the model has not seen.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(),
)
model.fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]
print(probability[:5].round(3))
print(y_valid.to_list()[:5])

**Example output**

```text
[0.177 0.017 0.53  0.007 0.062]
[0, 0, 1, 0, 0]
```

## A threshold turns a probability into a decision

*PowerPoint slide 12*

decision = alert when probability ≥ 0.5, otherwise no alert
Lowering the threshold raises more alerts. Raising it raises fewer.

## Loading the supplied observations

*PowerPoint slide 14*

raw_data: original CSV table; keep it unchanged so the analysis can be checked.

Run the cell and read the output below it.

In [ ]:
import pandas as pd
raw_data = pd.read_csv("classification_case.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)

## Every paired result belongs in one cell

*PowerPoint slide 15*

Read actual first, then predicted_values. Explain the missed warning.

## Counting the supplied paired labels

*PowerPoint slide 16*

actual holds the reference label. predicted holds the model decision. In this file, 1 means warning and 0 means no warning.

== compares values. & requires both conditions for each row. Parenthesise each comparison. sum counts the True entries.

## Operations for the next example

*PowerPoint slide 17*

01

Selecting columns: equipment_data["column"] returns one labelled column, a Series. equipment_data[["a", "b"]] returns a DataFrame containing those named columns in that order.

02

Column arithmetic: Arithmetic on columns calculates one result per row. Series align by row label. Divide only where the denominator is valid and keep actual and predicted observations paired.

03

Adding values and counting True: series.sum() adds its values. For a Boolean mask, True counts as 1 and False as 0. equipment_data.isna().sum() therefore counts missing cells separately for each column.

## Selecting rows with a mask

*PowerPoint slide 18*

A column comparison produces one True or False per row. table[mask] keeps the True rows. & combines masks: both conditions must hold. Parenthesise each comparison.

## Keep observations and decisions paired

*PowerPoint slide 19*

actual contains the recorded class; predicted contains the model’s decision for the same row.

Here 1 means positive and 0 negative.

These are already-issued decisions; this exercise does not train a classifier.

## Confusion counts from classification_case.csv: step 1

*PowerPoint slide 20*

actual_labels: the recorded class of each observation: 1 means positive and 0 means negative.

Run the cell and read the output below it.

In [ ]:
actual_labels = raw_data["actual"]
predicted_labels = raw_data["predicted"]

## Turn each class label into a question

*PowerPoint slide 21*

actual == 1 asks whether the real class is positive; predicted == 1 asks whether the model predicted positive.

Each comparison returns one True/False answer per row.

Equality uses ==; assignment uses =.

## Confusion counts from classification_case.csv: step 2

*PowerPoint slide 22*

actual_labels: the recorded class of each observation: 1 means positive and 0 means negative.

Run the cell and read the output below it.

In [ ]:
actual_is_positive = actual_labels == 1
prediction_is_positive = predicted_labels == 1
actual_is_negative = actual_labels == 0
prediction_is_negative = predicted_labels == 0
print(actual_is_positive.head())
print(prediction_is_positive.head())

## Count one outcome at a time

*PowerPoint slide 23*

A true positive has actual 1 AND predicted 1.

Combine the two answer columns with &, then sum the True entries.

The other three outcomes use their own pairs of actual and predicted labels.

## Confusion counts from classification_case.csv: step 3

*PowerPoint slide 24*

actual_labels: the recorded class of each observation: 1 means positive and 0 means negative.

Run the cell and read the output below it.

In [ ]:
is_true_positive = actual_is_positive & prediction_is_positive
true_positive_count = int(is_true_positive.sum())
print("True positives:", true_positive_count)

## Confusion counts from classification_case.csv: step 4

*PowerPoint slide 25*

false_negative_count: number of missed positives: actual 1 but predicted 0.

false_positive_count: number of false alarms: actual 0 but predicted 1.

## Confusion counts from classification_case.csv: step 4

*PowerPoint slide 28*

actual_labels: the recorded class of each observation: 1 means positive and 0 means negative.

Run the cell and read the output below it.

In [ ]:
false_negative_count = int(
    (actual_is_positive & prediction_is_negative).sum()
)
false_positive_count = int(
    (actual_is_negative & prediction_is_positive).sum()
)
true_negative_count = int(
    (actual_is_negative & prediction_is_negative).sum()
)
print(
    false_negative_count,
    false_positive_count,
    true_negative_count,
)
print(
    true_positive_count
    + false_negative_count
    + false_positive_count
    + true_negative_count,
    len(raw_data),
)

## Precision, recall and cost

*PowerPoint slide 29*

Precision is TP divided by TP + FP. Recall is TP divided by TP + FN. The issued policy returns zero when a denominator is zero.

The teaching cost is five units per missed warning plus one per false alarm. Explain the assumption when interpreting the result.

## Operations for the next example

*PowerPoint slide 30*

Rounded display: series.round(3) returns values rounded to three decimal places. Python round(number, 3) rounds one number. Keep unrounded calculations for returned assessment answers.

## Checking a condition before calculating

*PowerPoint slide 31*

if condition: runs its indented statements only when the condition is True. else: handles the other case. Check for a nonzero denominator before division, and for a nonempty table before selecting its first row.

## Precision: how trustworthy are positive predictions?

*PowerPoint slide 32*

The denominator is all predicted positives: true positives plus false positives.

Divide correct positive predictions by that total.

Check the denominator before dividing; this teaching example uses 0 when it is zero.

## Metrics from the supplied outcomes: step 1

*PowerPoint slide 33*

precision: fraction of positive predictions that are correct.

Run the cell and read the output below it.

In [ ]:
precision = 0.0
if true_positive_count + false_positive_count > 0:
    precision = true_positive_count / (
        true_positive_count + false_positive_count
    )

## Recall: how many real positives were found?

*PowerPoint slide 34*

The denominator is all actual positives: true positives plus false negatives.

Divide detected positives by that total.

Precision and recall answer different questions even when they happen to have the same value.

## Metrics from the supplied outcomes: step 2

*PowerPoint slide 35*

recall: fraction of real positives that the model found.

Run the cell and read the output below it.

In [ ]:
recall = 0.0
if true_positive_count + false_negative_count > 0:
    recall = true_positive_count / (
        true_positive_count + false_negative_count
    )

## Metrics from the supplied outcomes: step 3

*PowerPoint slide 36*

total_error_cost: five penalty units per missed positive plus one per false alarm in this scenario.

## Metrics from the supplied outcomes: step 3

*PowerPoint slide 37*

total_error_cost: five units per missed alert plus one per false alarm.

Run the cell and read the output below it.

In [ ]:
total_error_cost = 5 * false_negative_count + false_positive_count
print(round(precision, 4), round(recall, 4), total_error_cost)

## The confusion matrix of our model on the validation rows

*PowerPoint slide 38*

15 validation rows. Accuracy 93 percent, precision 0.75, recall 1.00.

## F1 combines precision and recall in one number

*PowerPoint slide 39*

F1 = 2 × precision × recall ÷ (precision + recall)

F1 is high only when precision and recall are both high.
Our model: precision 0.75, recall 1.00, F1 0.86.
A model that never raises an alert has recall 0, so its F1 is 0, even though its accuracy looked high.

## Moving the threshold trades false alarms for missed alerts

*PowerPoint slide 40*

Lower threshold: more alerts, more false alarms, fewer misses.
Higher threshold: fewer alerts, fewer false alarms, more misses.
Choose the threshold from the cost of each mistake.

## The ROC curve shows every threshold at once

*PowerPoint slide 41*

Each point is one threshold. AUC is 0.97 here. 1.00 is a perfect ranking and 0.50 is guessing.

## A decision tree can answer yes or no too

*PowerPoint slide 42*

SAME IDEA AS WEEK 8
A tree asks yes or no questions about the inputs. For a classifier, each leaf holds the share of alerts among the training rows that end there, and the tree answers with the majority.
In scikit-learn: DecisionTreeClassifier(max_depth=3, random_state=17).

ON THE VALIDATION ROWS
Tree: TP 3, FP 1, FN 0, TN 11. Precision 0.75, recall 1.00, F1 0.86.
Logistic regression: TP 3, FP 1, FN 0, TN 11. Precision 0.75, recall 1.00, F1 0.86.

## Your issued task

*PowerPoint slide 44*

01

Use classification_case.csv.

02

Positive means a synthetic warning condition.

03

Compute the confusion matrix, precision, recall and total_error_cost.

04

Explain the 5:1 missed-warning/false-alarm cost assumption and why accuracy alone is insufficient.

## Your issued task continued

*PowerPoint slide 45*

01

These costs are teaching assumptions for practice.

## classification_metrics(actual, predicted)

*PowerPoint slide 46*

Return dictionary keys tp, tn, fp, fn, precision, recall and cost.

tp = true positives; tn = true negatives; fp = false positives; fn = false negatives.

cost = 5 * fn + fp. Return zero when a precision or recall denominator is zero.

Keep the exact names and outputs. Check types, units and the issued brief.

## Written evidence

*PowerPoint slide 47*

01

Use the cell headed “My written evidence”.

02

Explain the supplied scenario in your own words.

03

State target units, reasons and limitations.

04

Submit one complete notebook containing code and written evidence.